# 13 · 富途 OpenD 取数（A股 / 港股 / 美股）

minibt 的股票数据源除了 `akshare / baostock / pytdx`，现在还支持 **富途 OpenD**：
数据来自本机 **FutuOpenD 网关**（默认 `127.0.0.1:11111`），**一个网关同时覆盖
A股 / 港股 / 美股**。

> 📘 对应文档：[9.8 数据源 —— 富途 OpenD](../../minibt_docs/docs/minibt_data/9.8_datasource.md)

## 前置条件

1. `pip install futu-api`
2. 下载并启动 **FutuOpenD**，登录富途账号，让窗口底部显示「已连接」
3. 在富途开通对应市场的**行情权限**（A股可在富途牛牛免费申请；港股/美股看账户）

> ⚠️ 连不上 OpenD 时，minibt 会**直接抛错**并给出提示（不会静默返回空数据）。

## 两种用法

```python
strategy.kline_from_futu('600000', 86400, 500)                  # 便捷函数
strategy.get_kline('600000', 86400, 500, data_source='futu')    # 统一入口
```

| 你写的代码 | futu 代码 |
| --- | --- |
| `600000` / `600000.SH` | `SH.600000` |
| `000001` | `SZ.000001` |
| `00700` | `HK.00700` |
| `AAPL` | `US.AAPL` |

周期（秒）：`60 / 300 / 900 / 1800 / 3600 / 86400 / 604800 / 2419200`
= 1 分 / 5 分 / 15 分 / 30 分 / 60 分 / 日 / 周 / 月。


## 0 · 环境检查（FutuOpenD 先连上）


In [ ]:
import os
import socket
import sys

# 找到仓库根目录（cwd 可能是仓库根，也可能在 tutorials/data 下）
for _path in (os.getcwd(), os.path.dirname(os.getcwd()),
              os.path.dirname(os.path.dirname(os.getcwd()))):
    if os.path.isdir(os.path.join(_path, 'minibt')) and _path not in sys.path:
        sys.path.insert(0, _path)

from minibt import *

HOST, PORT = '127.0.0.1', 11111
try:
    with socket.create_connection((HOST, PORT), timeout=3):
        print(f'✅ FutuOpenD 已连接：{HOST}:{PORT}')
except OSError as error:
    print(f'❌ 连不上 FutuOpenD（{HOST}:{PORT}）：{error}')
    print('   请先启动并登录 FutuOpenD，再重新运行本 notebook。')


## 1 · 取 A 股日线（最简单）

`kline_from_futu(代码, 周期秒, 根数)` → 返回 `KLine`（和 CSV / tqsdk 拿到的完全一样，
可以直接算指标、回测、画图）。


In [ ]:
strategy = Strategy()

kline = strategy.kline_from_futu('600000', duration_seconds=86400,
                                 data_length=500)
print('KLine:', len(kline), '根 |', kline.symbol, '| cycle', kline.cycle)

df = kline.pandas_object
print('列:', list(df.columns))
df.tail()


## 2 · 算指标 + 出图

`KLine` 上的 `run(...)` 会直接渲染交互图（Jupyter 里内嵌显示）。


In [ ]:
kline.run(
    ["ma5", Multiply(PandasTa.sma, dict(length=5))],
    ["ma20", Multiply(PandasTa.sma, dict(length=20))],
    ["ma60", Multiply(PandasTa.sma, dict(length=60))],
)


## 3 · 分钟线 / 港股 / 美股

同一个网关取不同市场；**没有行情权限的会抛错**（下面是 `try/except` 演示）。


In [ ]:
# 同一个网关取不同市场/周期；没有权限或**额度用尽**都会抛错（这里 try 住演示）
targets = [
    ('600519', 300, 'A股 5 分钟'),
    ('00700', 86400, '港股 日线'),
    ('AAPL', 86400, '美股 日线'),
]
for code, cycle, label in targets:
    try:
        item = strategy.kline_from_futu(code, duration_seconds=cycle,
                                        data_length=200)
        print(f'{label:10} {code} → {len(item)} 根 | {item.symbol}')
    except Exception as error:                 # noqa: BLE001
        print(f'{label:10} {code} ✗ {type(error).__name__}: '
              f'{str(error).splitlines()[0]}')


## 4 · 统一入口 + 复权参数

`get_kline(..., data_source='futu')` 与 `kline_from_futu` 等价，
另外支持 `adjust_type`：`'qfq'`（前复权，默认）/ `'hfq'`（后复权）/ `'none'`。


In [ ]:
kline2 = strategy.get_kline('600000', duration_seconds=86400,
                            data_length=800, data_source='futu',
                            adjust_type='qfq')
print('get_kline(data_source="futu"):', len(kline2), '根 |', kline2.symbol)

# 顺手存成 CSV —— 下次离线回测直接用
path = '600000_futu.csv'
kline2.pandas_object.to_csv(path, index=False)
print('已保存:', os.path.abspath(path))


## 5 · 没启动 OpenD 会怎样？

把端口改成一个没人监听的端口，看看报错（**不会静默返回空数据**）。


In [ ]:
try:
    strategy.kline_from_futu('600000', duration_seconds=86400,
                             data_length=10, port=1, timeout=1)
except RuntimeError as error:
    print('✅ 预期内的报错：')
    print(error)


## 6 · 额度与常见问题

| 现象 | 原因 / 解决 |
| --- | --- |
| `未检测到 FutuOpenD 网关` | OpenD 没启动 / 端口不对（OpenD 界面底部显示端口，默认 11111） |
| `未安装 futu-api` | `pip install futu-api` |
| `历史K线额度不足（已用正股额度：100/100）` | 额度按**标的**计：100 只股票 / 7 天滚动（期权 20 只） |
| `行情权限不足` | 该市场没订阅行情（富途行情商城 / 牛牛里申请） |
| 数据只有几百根 | `get_kline` 回测模式最小 300 根；超过可用历史时只返回实际有的根数 |

### 额度怎么算（**按标的，不按周期**）

* 取一只**新**股票（日线或分钟线都一样）→ 消耗 1 个额度；
* **同一只股票重复取**（换成 5 分钟、周线…）→ **不额外消耗**；
* 额度用尽 → `历史K线额度不足（已用正股额度：100/100）`，**7 天滚动释放**。

> ⚠️ 「A股日线免费」是误解 —— 免费的是**已经取过的那只标的**，与周期无关。
> 额度满时实测：`600000` 的日线 / 5 分钟都能取，而 `600519 / 000001 / 00700 /
> AAPL`（新标的）全部报 100/100。

### 省额度的做法

* 回测批量取数优先用 `akshare / baostock`（A股免费、不限标的数）；
* 必须用 futu 时：**一次取好存 CSV**（`kline.pandas_object.to_csv(...)` 或
  `get_kline(..., save='xxx')`），以后离线复用；
* 实测脚本：`tutorials/data/14_futu_quota.py`。
